### Frame Extraction

In [1]:
import av
import os
import numpy as np
from PIL import Image
from tqdm import tqdm

CHOLEC80_DIR = "Cholec80/cholec80"
VIDEO_DIR = os.path.join(CHOLEC80_DIR, "videos")
PHASE_DIR = os.path.join(CHOLEC80_DIR, "phase_annotations")
FRAME_DIR = "cholec80_frames"   # extracted frames go here
EXTRACT_FPS = 1         # 1 frame per second
SAMPLE_EVERY = 25       # 25fps/1fps = every 25th frame

PHASE_MAP = {
    'Preparation':             0,
    'CalotTriangleDissection': 1,
    'ClippingCutting':         2,
    'GallbladderDissection':   3,
    'GallbladderPackaging':    4,
    'CleaningCoagulation':     5,
    'GallbladderRetraction':   6,
}

In [2]:
# frame number -> phase lebel (int)
def parse_phase_annotation(video_id):
    path = os.path.join(PHASE_DIR, f"video{video_id:02d}-phase.txt")
    labels = {}

    with open(path) as f:
        next(f)     # skip header
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) == 2:
                frame_num = int(parts[0])
                phase_str = parts[1].strip()
                if phase_str in PHASE_MAP:
                    labels[frame_num] = PHASE_MAP[phase_str]
            
    return labels

In [3]:
# extract frames
extraction_log = {}     #   video_id -> list of (frame_path, phase_label)

for video_id in tqdm(range(1, 81), desc="Videos"):
    video_path = os.path.join(VIDEO_DIR, f"video{video_id:02d}.mp4")
    if not os.path.exists(video_path):
        print(f"Missing: video{video_id:02d}.mp4 - skipping")
        continue

    # load phase labels for this video
    phase_labels = parse_phase_annotation(video_id)

    # output folder per video
    vid_frame_dir = os.path.join(FRAME_DIR, f"video{video_id:02d}")
    os.makedirs(vid_frame_dir, exist_ok=True)

    video_log = []
    container = av.open(video_path)

    for frame_idx, frame in enumerate(container.decode(video=0)):
        # extract 1fps from 25fps (every 25th frame)
        if frame_idx % SAMPLE_EVERY != 0:
            continue

        # get phase label for this phrame
        phase = phase_labels.get(frame_idx, -1)
        if phase == -1:
            continue # no annotations for this frame

        # save as JPEG

        img = frame.to_ndarray(format='rgb24')
        img_pil = Image.fromarray(img)
        img_small = img_pil.resize((224,224), Image.LANCZOS)

        frame_name = f"f{frame_idx:06d}.jpg"
        frame_path = os.path.join(vid_frame_dir, frame_name)
        img_small.save(frame_path, quality=90)

        video_log.append((frame_path, phase))

    container.close()
    extraction_log[video_id] = video_log

total_frames = sum(len(v) for v in extraction_log.values())
print(f"\n{'='*50}")
print(f"Extraction complete")
print(f"Videos processed:  {len(extraction_log)}")
print(f"Total frames:      {total_frames:,}")
print(f"\nPhase distribution:")

phase_counts = {v: 0 for v in PHASE_MAP.values()}
for video_log in extraction_log.values():
    for _, phase in video_log:
        phase_counts[phase] += 1

phase_names = {v: k for k, v in PHASE_MAP.items()}
for phase_id, count in sorted(phase_counts.items()):
    pct = count / total_frames * 100
    print(f"  {phase_names[phase_id]:<30} {count:>6,} frames ({pct:.1f}%)")

# save the log for later use
import json
log_serializable = {
    str(k): v for k, v in extraction_log.items()
}
with open("cholec80_frame_log.json", "w") as f:
    json.dump(log_serializable, f)
print(f"\nFrame log saved to: cholec80_frame_log.json")
print(f"Disk usage: ~{total_frames * 50 / 1024 / 1024:.1f} MB estimated")

Videos:   0%|          | 0/80 [00:30<?, ?it/s]


KeyboardInterrupt: 

### DINO VIT based feature extraction

In [4]:
import os
import json
import numpy as np
import torch
import timm
from PIL import Image
from torchvision import transforms
from tqdm import tqdm

FRAME_DIR    = "cholec80_frames"
FEATURE_DIR  = "cholec80_features_dino"
BATCH_SIZE   = 64
device       = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

os.makedirs(FEATURE_DIR, exist_ok=True)

In [5]:
print("Loading DINO-ViT-Base teacher...")
teacher = timm.create_model(
    'vit_base_patch16_224.dino',
    pretrained=True,
    num_classes=0        # remove classification head — features only
)
teacher = teacher.to(device)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

print(f"  Teacher loaded — output dim: {teacher.num_features}")
print(f"  Parameters: {sum(p.numel() for p in teacher.parameters()):,}")

Loading DINO-ViT-Base teacher...


  Teacher loaded — output dim: 768
  Parameters: 85,798,656


In [6]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224), antialias=True),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

with open("cholec80_frame_log.json") as f:
    frame_log = json.load(f)

print(f"  Frame log loaded — {sum(len(v) for v in frame_log.values()):,} frames")


  Frame log loaded — 184,578 frames


In [7]:
print(f"\nExtracting DINO features → {FEATURE_DIR}/")
print("="*50)

total_extracted = 0

for video_id_str in tqdm(sorted(frame_log.keys(),
                                 key=lambda x: int(x)),
                          desc="Videos"):
    video_id  = int(video_id_str)
    video_log = frame_log[video_id_str]  # list of [frame_path, phase_label]

    if len(video_log) == 0:
        continue

    out_path = os.path.join(FEATURE_DIR, f"video{video_id:02d}.npz")
    if os.path.exists(out_path):
        total_extracted += len(video_log)
        continue   # already done — resume friendly

    frame_paths   = [entry[0] for entry in video_log]
    phase_labels  = [entry[1] for entry in video_log]

    all_features = []

    # process in batches
    for batch_start in range(0, len(frame_paths), BATCH_SIZE):
        batch_paths = frame_paths[batch_start:batch_start + BATCH_SIZE]

        # load and transform batch
        imgs = []
        for path in batch_paths:
            img = Image.open(path).convert('RGB')
            imgs.append(transform(img))

        batch_tensor = torch.stack(imgs).to(device)  # (B, 3, 224, 224)

        with torch.no_grad():
            feats = teacher(batch_tensor)             # (B, 768)

        all_features.append(feats.cpu().numpy())

    # save features + labels together
    features_np = np.concatenate(all_features, axis=0)  # (N, 768)
    labels_np   = np.array(phase_labels)                 # (N,)

    np.savez_compressed(out_path,
                        features=features_np,
                        labels=labels_np,
                        frame_paths=np.array(frame_paths))

    total_extracted += len(video_log)

print(f"\n{'='*50}")
print(f"Feature extraction complete")
print(f"Total frames processed: {total_extracted:,}")

# check one file
sample = np.load(os.path.join(FEATURE_DIR, "video01.npz"))
print(f"\nSample — video01.npz:")
print(f"  features shape: {sample['features'].shape}")
print(f"  labels shape:   {sample['labels'].shape}")
print(f"  feature dim:    {sample['features'].shape[1]}")
print(f"  label range:    {sample['labels'].min()} to {sample['labels'].max()}")

# check disk usage
total_size = sum(
    os.path.getsize(os.path.join(FEATURE_DIR, f))
    for f in os.listdir(FEATURE_DIR)
)
print(f"\nFeature storage: {total_size / 1024**2:.1f} MB")


Extracting DINO features → cholec80_features_dino/


Videos: 100%|██████████| 80/80 [00:00<00:00, 35084.10it/s]


Feature extraction complete
Total frames processed: 184,578

Sample — video01.npz:
  features shape: (1734, 768)
  labels shape:   (1734,)
  feature dim:    768
  label range:    0 to 6

Feature storage: 504.0 MB


### Training

In [8]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

FEATURE_DIR   = "cholec80_features_dino"
CHECKPOINT_DIR = "stage8_checkpoints"
TRAIN_VIDEOS  = list(range(1, 41))    # videos 1-40 for pretraining
VAL_VIDEOS    = list(range(41, 81))   # videos 41-80 held out
BATCH_SIZE    = 256                    # large batch — features are tiny
LR            = 3e-4
N_EPOCHS      = 20
FEAT_DIM      = 384                    # student ViT-Small output
TEACHER_DIM   = 768                    # DINO-ViT-Base output
SEED          = 42
device        = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
torch.manual_seed(SEED)

In [9]:
# dataset
class CholecFeatureDataset(Dataset):
    
    # Loads pre-extracted DINO teacher features from disk.
    # Returns (teacher_feature, phase_label) per frame.
    # No images loaded during training — very fast.
    
    def __init__(self, video_ids, feature_dir):
        self.features = []
        self.labels   = []

        for vid_id in video_ids:
            path = os.path.join(feature_dir, f"video{vid_id:02d}.npz")
            if not os.path.exists(path):
                print(f"  Missing: video{vid_id:02d}.npz — skipping")
                continue
            data = np.load(path)
            self.features.append(data['features'])  # (N, 768)
            self.labels.append(data['labels'])       # (N,)

        self.features = np.concatenate(self.features, axis=0)  # (total, 768)
        self.labels   = np.concatenate(self.labels,   axis=0)  # (total,)

        print(f"  Loaded {len(self.features):,} frames from {len(video_ids)} videos")

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        feat  = torch.tensor(self.features[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx],   dtype=torch.long)
        return feat, label


In [10]:
class StudentEncoder(nn.Module):
    
    def __init__(self, teacher_dim=768, student_dim=384):
        super().__init__()

        # projection head — maps teacher features to student space
        # then back to teacher space for distillation loss
        self.encoder = nn.Sequential(
            nn.Linear(teacher_dim, 512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Linear(512, student_dim),
            nn.GELU(),
            nn.LayerNorm(student_dim),
        )

        # projection back to teacher dim for loss computation
        self.proj = nn.Linear(student_dim, teacher_dim)

    def forward(self, x):
        h    = self.encoder(x)   # (B, student_dim)
        proj = self.proj(h)      # (B, teacher_dim)
        return h, proj


# ── Loss ──────────────────────────────────────────────────────────────────────
def distillation_loss(student_proj, teacher_feat):
    """
    Cosine distance between student projection and teacher features.
    0 = identical, 2 = opposite.
    """
    s = F.normalize(student_proj, dim=-1)
    t = F.normalize(teacher_feat, dim=-1)
    return (2 - 2 * (s * t).sum(dim=-1)).mean()


print("Stage 8 — Cholec80 Pretraining via DINO Distillation")
print("="*55)
print("\nLoading datasets...")

train_ds = CholecFeatureDataset(TRAIN_VIDEOS, FEATURE_DIR)
val_ds   = CholecFeatureDataset(VAL_VIDEOS,   FEATURE_DIR)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)

print(f"\nTrain: {len(train_ds):,} frames | Val: {len(val_ds):,} frames")

model     = StudentEncoder(teacher_dim=TEACHER_DIM,
                           student_dim=FEAT_DIM).to(device)
optimizer = torch.optim.AdamW(model.parameters(),
                               lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS, eta_min=1e-6
)

trainable = sum(p.numel() for p in model.parameters())
print(f"Student parameters: {trainable:,}")

Stage 8 — Cholec80 Pretraining via DINO Distillation

Loading datasets...
  Loaded 86,344 frames from 40 videos
  Loaded 98,234 frames from 40 videos

Train: 86,344 frames | Val: 98,234 frames
Student parameters: 888,192


In [11]:
print(f"\nTraining for {N_EPOCHS} epochs...")
print("="*55)

best_val_loss = float('inf')

for epoch in range(1, N_EPOCHS + 1):
    import time
    t0 = time.time()

    # train
    model.train()
    train_losses = []
    for teacher_feat, _ in train_loader:   # labels not used in pretraining
        teacher_feat = teacher_feat.to(device)
        h, proj      = model(teacher_feat)
        loss         = distillation_loss(proj, teacher_feat)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    scheduler.step()

    # validate
    model.eval()
    val_losses = []
    with torch.no_grad():
        for teacher_feat, _ in val_loader:
            teacher_feat = teacher_feat.to(device)
            h, proj      = model(teacher_feat)
            loss         = distillation_loss(proj, teacher_feat)
            val_losses.append(loss.item())

    train_loss = np.mean(train_losses)
    val_loss   = np.mean(val_losses)
    elapsed    = time.time() - t0

    print(f"  Epoch {epoch:2d} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"{elapsed:.1f}s")

    # save best
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch':            epoch,
            'model_state_dict': model.state_dict(),
            'best_val_loss':    best_val_loss,
            'train_loss':       train_loss,
        }, os.path.join(CHECKPOINT_DIR, 'stage8_best.pt'))

# save final
torch.save({
    'epoch':            N_EPOCHS,
    'model_state_dict': model.state_dict(),
    'final_val_loss':   val_loss,
}, os.path.join(CHECKPOINT_DIR, 'stage8_final.pt'))

print(f"\n{'='*55}")
print(f"Pretraining complete")
print(f"Best val loss: {best_val_loss:.4f}")
print(f"Checkpoints saved to: {CHECKPOINT_DIR}/")


Training for 20 epochs...
  Epoch  1 | Train Loss: 0.2977 | Val Loss: 0.1609 | 2.6s
  Epoch  2 | Train Loss: 0.1135 | Val Loss: 0.1044 | 2.4s
  Epoch  3 | Train Loss: 0.0808 | Val Loss: 0.0828 | 2.4s
  Epoch  4 | Train Loss: 0.0665 | Val Loss: 0.0715 | 2.6s
  Epoch  5 | Train Loss: 0.0586 | Val Loss: 0.0652 | 2.4s
  Epoch  6 | Train Loss: 0.0535 | Val Loss: 0.0600 | 2.4s
  Epoch  7 | Train Loss: 0.0501 | Val Loss: 0.0570 | 2.4s
  Epoch  8 | Train Loss: 0.0476 | Val Loss: 0.0547 | 2.3s
  Epoch  9 | Train Loss: 0.0458 | Val Loss: 0.0531 | 2.4s
  Epoch 10 | Train Loss: 0.0444 | Val Loss: 0.0516 | 2.3s
  Epoch 11 | Train Loss: 0.0434 | Val Loss: 0.0507 | 2.9s
  Epoch 12 | Train Loss: 0.0426 | Val Loss: 0.0498 | 2.4s
  Epoch 13 | Train Loss: 0.0419 | Val Loss: 0.0492 | 2.4s
  Epoch 14 | Train Loss: 0.0414 | Val Loss: 0.0487 | 2.3s
  Epoch 15 | Train Loss: 0.0410 | Val Loss: 0.0483 | 2.4s
  Epoch 16 | Train Loss: 0.0407 | Val Loss: 0.0480 | 2.3s
  Epoch 17 | Train Loss: 0.0405 | Val Loss: 0

In [12]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

# load best checkpoint
ckpt = torch.load('stage8_checkpoints/stage8_best.pt',
                   map_location=device,
                   weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

# extract student embeddings for val set
print("Extracting student embeddings for val set...")
all_h      = []
all_labels = []

with torch.no_grad():
    for teacher_feat, labels in val_loader:
        teacher_feat = teacher_feat.to(device)
        h, _         = model(teacher_feat)   # (B, 384) student embedding
        all_h.append(h.cpu().numpy())
        all_labels.extend(labels.numpy())

X = np.concatenate(all_h,  axis=0)   # (N_val, 384)
y = np.array(all_labels)              # (N_val,)

# linear probe — can student embeddings predict phase?
print("Running linear probe on student embeddings...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# use first half to train probe, second half to test
split = len(X) // 2
X_tr, X_te = X_scaled[:split], X_scaled[split:]
y_tr, y_te = y[:split],        y[split:]

clf = LogisticRegression(max_iter=1000, C=1.0)
clf.fit(X_tr, y_tr)
preds = clf.predict(X_te)

f1 = f1_score(y_te, preds, average='macro', zero_division=0)

PHASE_NAMES = {
    0: 'Preparation',
    1: 'CalotTriangleDissection',
    2: 'ClippingCutting',
    3: 'GallbladderDissection',
    4: 'GallbladderPackaging',
    5: 'CleaningCoagulation',
    6: 'GallbladderRetraction',
}

print(f"\nLinear probe on student embeddings:")
print(f"  Macro F1: {f1:.3f}")
print(f"\nPer-class F1:")
f1_per = f1_score(y_te, preds, average=None, zero_division=0)
for i, score in enumerate(f1_per):
    print(f"  {PHASE_NAMES[i]:<30} {score:.3f}")

print(f"\nBaseline (random):     ~0.143  (1/7 classes)")
print(f"Stage 7 scratch (100%): 0.609")
print(f"Student linear probe:   {f1:.3f}")

Extracting student embeddings for val set...
Running linear probe on student embeddings...

Linear probe on student embeddings:
  Macro F1: 0.623

Per-class F1:
  Preparation                    0.629
  CalotTriangleDissection        0.769
  ClippingCutting                0.525
  GallbladderDissection          0.777
  GallbladderPackaging           0.670
  CleaningCoagulation            0.455
  GallbladderRetraction          0.540

Baseline (random):     ~0.143  (1/7 classes)
Stage 7 scratch (100%): 0.609
Student linear probe:   0.623


In [13]:
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

#  Load all val embeddings (already extracted above)
# X and y are already in memory from the previous cell

# Also extract train embeddings 
print("Extracting student embeddings for train set...")
all_h_train      = []
all_labels_train = []

train_ds_eval = CholecFeatureDataset(TRAIN_VIDEOS, FEATURE_DIR)
train_loader_eval = DataLoader(train_ds_eval, batch_size=256,
                                shuffle=False, num_workers=0)

model.eval()
with torch.no_grad():
    for teacher_feat, labels in train_loader_eval:
        teacher_feat = teacher_feat.to(device)
        h, _         = model(teacher_feat)
        all_h_train.append(h.cpu().numpy())
        all_labels_train.extend(labels.numpy())

X_train = np.concatenate(all_h_train,  axis=0)
y_train = np.array(all_labels_train)

print(f"  Train embeddings: {X_train.shape}")
print(f"  Val embeddings:   {X.shape}")

#  Label efficiency sweep
print("\nLabel efficiency — Student vs Random baseline")
print("="*55)
print(f"  {'Data %':<10} {'Student F1':>12} {'Random F1':>12} {'Gain':>8}")
print(f"  {'-'*44}")

scaler   = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_train)
X_te_sc  = scaler.transform(X)

results = {}

for pct in [10, 25, 50, 100]:
    # subsample training data
    n_train = int(len(X_train) * pct / 100)
    
    # stratified subsample — keep class proportions
    indices = []
    for cls in range(7):
        cls_idx = np.where(y_train == cls)[0]
        n_cls   = max(1, int(len(cls_idx) * pct / 100))
        chosen  = np.random.choice(cls_idx, n_cls, replace=False)
        indices.extend(chosen)
    indices = np.array(indices)

    X_sub = X_tr_sc[indices]
    y_sub = y_train[indices]

    # student linear probe
    clf_student = LogisticRegression(max_iter=1000, C=1.0)
    clf_student.fit(X_sub, y_sub)
    preds_student = clf_student.predict(X_te_sc)
    f1_student    = f1_score(y, preds_student,
                              average='macro', zero_division=0)

    # random baseline — random init embeddings (no pretraining)
    np.random.seed(42)
    X_random = np.random.randn(*X_train.shape).astype(np.float32)
    X_rand_sc = scaler.fit_transform(X_random)
    X_te_rand = scaler.transform(
        np.random.randn(*X.shape).astype(np.float32)
    )
    X_rand_sub = X_rand_sc[indices]
    clf_random = LogisticRegression(max_iter=1000, C=1.0)
    clf_random.fit(X_rand_sub, y_sub)
    preds_random = clf_random.predict(X_te_rand)
    f1_random    = f1_score(y, preds_random,
                             average='macro', zero_division=0)

    gain = f1_student - f1_random
    results[pct] = (f1_student, f1_random, gain)

    print(f"  {str(pct)+'%':<10} {f1_student:>12.3f} {f1_random:>12.3f} "
          f"{gain:>+8.3f}")

print(f"\n  Stage 7 scratch (100% labels, fine-tuned): 0.609")
print(f"  Student linear probe (100% labels):        "
      f"{results[100][0]:.3f}")
print(f"\n  Key finding: student beats scratch at what % of data?")
for pct, (f1_s, f1_r, gain) in results.items():
    if f1_s >= 0.609:
        print(f"  → Student matches Stage 7 scratch with only {pct}% of labels")
        break

Extracting student embeddings for train set...
  Loaded 86,344 frames from 40 videos
  Train embeddings: (86344, 384)
  Val embeddings:   (98234, 384)

Label efficiency — Student vs Random baseline
  Data %       Student F1    Random F1     Gain
  --------------------------------------------
  10%               0.565        0.118   +0.447
  25%               0.605        0.099   +0.506
  50%               0.614        0.088   +0.526
  100%              0.621        0.081   +0.540

  Stage 7 scratch (100% labels, fine-tuned): 0.609
  Student linear probe (100% labels):        0.621

  Key finding: student beats scratch at what % of data?
  → Student matches Stage 7 scratch with only 50% of labels


### Stage 8B

### Pre-extract ViT

In [20]:
import os
import numpy as np
import torch
import timm
from PIL import Image
from torchvision import transforms
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
FRAME_BASE    = "cholec80_frames"
VIT_FEAT_DIR  = "cholec80_features_vit_small"
BATCH_SIZE    = 128
device        = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

os.makedirs(VIT_FEAT_DIR, exist_ok=True)

# ── Load ViT-Small ────────────────────────────────────────────────────────────
print("Loading ViT-Small...")
vit_small = timm.create_model(
    'vit_small_patch16_224.dino',
    pretrained=True,
    num_classes=0
)
vit_small = vit_small.to(device)
vit_small.eval()
for p in vit_small.parameters():
    p.requires_grad = False
print(f"  ViT-Small loaded — output dim: {vit_small.num_features}")

# ── Transform ─────────────────────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224), antialias=True),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ── Extract per video ─────────────────────────────────────────────────────────
print(f"\nExtracting ViT-Small features → {VIT_FEAT_DIR}/")
print("="*50)

for video_id in tqdm(range(1, 81), desc="Videos"):
    out_path = os.path.join(VIT_FEAT_DIR, f"video{video_id:02d}.npz")
    if os.path.exists(out_path):
        continue   # resume friendly

    frame_dir = os.path.join(FRAME_BASE, f"video{video_id:02d}")
    if not os.path.exists(frame_dir):
        continue

    # load all frames for this video sorted by frame number
    fnames = sorted(
        f for f in os.listdir(frame_dir) if f.endswith('.jpg')
    )
    frame_nums = [int(f.replace('f','').replace('.jpg',''))
                  for f in fnames]

    all_features = []

    # process in batches
    for batch_start in range(0, len(fnames), BATCH_SIZE):
        batch_fnames = fnames[batch_start:batch_start + BATCH_SIZE]
        imgs = []
        for fname in batch_fnames:
            img = Image.open(
                os.path.join(frame_dir, fname)
            ).convert('RGB')
            imgs.append(transform(img))

        batch_tensor = torch.stack(imgs).to(device)  # (B, 3, 224, 224)

        with torch.no_grad():
            feats = vit_small(batch_tensor)           # (B, 384)

        all_features.append(feats.cpu().numpy())

    features_np  = np.concatenate(all_features, axis=0)  # (N, 384)
    frame_nums_np = np.array(frame_nums)                   # (N,)

    np.savez_compressed(out_path,
                        features=features_np,
                        frame_nums=frame_nums_np)

# ── Summary ───────────────────────────────────────────────────────────────────
sample = np.load(os.path.join(VIT_FEAT_DIR, "video01.npz"))
print(f"\nExtraction complete")
print(f"Sample video01 — features: {sample['features'].shape}")
print(f"Feature dim: {sample['features'].shape[1]}")

total_size = sum(
    os.path.getsize(os.path.join(VIT_FEAT_DIR, f))
    for f in os.listdir(VIT_FEAT_DIR)
    if f.endswith('.npz')
)
print(f"Storage: {total_size / 1024**2:.1f} MB")

Loading ViT-Small...
  ViT-Small loaded — output dim: 384

Extracting ViT-Small features → cholec80_features_vit_small/


Videos: 100%|██████████| 80/80 [18:03<00:00, 13.54s/it]


Extraction complete
Sample video01 — features: (1734, 384)
Feature dim: 384
Storage: 251.3 MB


#### Dataset

In [25]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
import time

# ── Config ────────────────────────────────────────────────────────────────────
VIT_FEAT_DIR   = "cholec80_features_vit_small"
DINO_FEAT_DIR  = "cholec80_features_dino"
CHECKPOINT_DIR = "stage8b_checkpoints"
TRAIN_VIDEOS   = list(range(1, 41))
VAL_VIDEOS     = list(range(41, 81))
CLIP_LEN       = 8
STEP           = 4
BATCH_SIZE     = 64    # larger now — no ViT forward pass
LR             = 1e-4
N_EPOCHS       = 30
FEAT_DIM       = 384   # ViT-Small output
TEACHER_DIM    = 768   # DINO-ViT-Base output
DISTILL_WEIGHT = 0.3
CLASS_WEIGHT   = 1.0
SEED           = 42
device         = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
torch.manual_seed(SEED)

# ── Dataset ───────────────────────────────────────────────────────────────────
class CholecPretrainedDataset(Dataset):
    """
    Loads pre-extracted ViT-Small features + DINO features.
    No images, no ViT forward pass during training.
    Returns (vit_clip, dino_clip, label) per clip.
    """
    def __init__(self, video_ids, vit_feat_dir,
                 dino_feat_dir, clip_len=8, step=4):
        self.clips      = []
        self.vit_cache  = {}
        self.dino_cache = {}
        self.label_cache = {}
        self.clip_len   = clip_len

        print(f"  Loading features for {len(video_ids)} videos...")
        for vid_id in video_ids:
            vit_path  = os.path.join(vit_feat_dir,
                                      f"video{vid_id:02d}.npz")
            dino_path = os.path.join(dino_feat_dir,
                                      f"video{vid_id:02d}.npz")
            if not os.path.exists(vit_path) or \
               not os.path.exists(dino_path):
                continue

            vit_data  = np.load(vit_path)
            dino_data = np.load(dino_path)

            vit_feats  = vit_data['features'].astype(np.float32)   # (N, 384)
            dino_feats = dino_data['features'].astype(np.float32)  # (N, 768)
            labels     = dino_data['labels'].astype(np.int64)       # (N,)

            self.vit_cache[vid_id]   = vit_feats
            self.dino_cache[vid_id]  = dino_feats
            self.label_cache[vid_id] = labels

            # build clips
            n    = len(labels)
            half = clip_len // 2
            for center in range(half, n - half, step):
                self.clips.append((
                    vid_id,
                    center - half,
                    center + half,
                    int(labels[center])
                ))

        print(f"  {len(self.clips):,} clips ready")

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, idx):
        video_id, start, end, label = self.clips[idx]

        # ViT-Small features for this clip
        vit_clip = torch.from_numpy(
            self.vit_cache[video_id][start:end+1]
        )
        while vit_clip.shape[0] < self.clip_len:
            vit_clip = torch.cat([vit_clip, vit_clip[-1:]], dim=0)
        vit_clip = vit_clip[:self.clip_len]   # (T, 384)

        # DINO features for distillation
        dino_clip = torch.from_numpy(
            self.dino_cache[video_id][start:end+1]
        )
        while dino_clip.shape[0] < self.clip_len:
            dino_clip = torch.cat([dino_clip, dino_clip[-1:]], dim=0)
        dino_clip = dino_clip[:self.clip_len]  # (T, 768)

        return vit_clip, dino_clip, torch.tensor(label)


#### Model

In [26]:
class TemporalClassifier(nn.Module):
    """
    Temporal Transformer on top of frozen ViT-Small features.
    Same architecture as Stage 3/4 Temporal Transformer.
    """
    def __init__(self, feat_dim=384, n_classes=7,
                 clip_len=8, n_heads=6, n_layers=4):
        super().__init__()

        self.cls_token = nn.Parameter(torch.randn(1, 1, feat_dim))
        self.pos_embed = nn.Parameter(
            torch.randn(1, clip_len + 1, feat_dim)
        )
        encoder_layer  = nn.TransformerEncoderLayer(
            d_model=feat_dim, nhead=n_heads,
            dim_feedforward=feat_dim * 4,
            dropout=0.3, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=n_layers
        )
        self.norm = nn.LayerNorm(feat_dim)
        self.head = nn.Linear(feat_dim, n_classes)

        # projection to match DINO-Base dim for distillation
        self.distill_proj = nn.Linear(feat_dim, 768)

    def forward(self, vit_feats):
        """
        vit_feats: (B, T, 384) — pre-extracted ViT-Small features
        returns: logits (B, 7), h (B, 384), proj (B, T, 768)
        """
        B = vit_feats.shape[0]

        # temporal transformer
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, vit_feats], dim=1)  # (B, T+1, 384)
        x   = x + self.pos_embed
        x   = self.transformer(x)
        x   = self.norm(x)

        h      = x[:, 0, :]                        # CLS token (B, 384)
        logits = self.head(h)                       # (B, 7)

        # per-frame projection for distillation
        frame_feats = x[:, 1:, :]                  # (B, T, 384)
        proj        = self.distill_proj(frame_feats) # (B, T, 768)

        return logits, h, proj


#### Data prep

In [27]:
print("Loading Stage 8 MLP teacher...")
mlp_teacher = StudentEncoder(
    teacher_dim=768, student_dim=384
).to(device)
ckpt_mlp = torch.load(
    'stage8_checkpoints/stage8_best.pt',
    map_location=device, weights_only=False
)
mlp_teacher.load_state_dict(ckpt_mlp['model_state_dict'])
mlp_teacher.eval()
for p in mlp_teacher.parameters():
    p.requires_grad = False
print("  MLP teacher loaded and frozen")

# ── Build datasets ────────────────────────────────────────────────────────────
print("\nBuilding datasets...")
train_ds = CholecPretrainedDataset(
    TRAIN_VIDEOS, VIT_FEAT_DIR, DINO_FEAT_DIR,
    clip_len=CLIP_LEN, step=STEP
)
val_ds = CholecPretrainedDataset(
    VAL_VIDEOS, VIT_FEAT_DIR, DINO_FEAT_DIR,
    clip_len=CLIP_LEN, step=STEP
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                           shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=0)

# ── Build model ───────────────────────────────────────────────────────────────
model8b   = TemporalClassifier(
    feat_dim=FEAT_DIM, n_classes=7, clip_len=CLIP_LEN
).to(device)
optimizer = torch.optim.AdamW(
    model8b.parameters(), lr=LR, weight_decay=1e-2
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS, eta_min=1e-6
)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

trainable = sum(p.numel() for p in model8b.parameters()
                if p.requires_grad)
print(f"\nTrainable parameters: {trainable:,}")

Loading Stage 8 MLP teacher...
  MLP teacher loaded and frozen

Building datasets...
  Loading features for 40 videos...
  21,522 clips ready
  Loading features for 40 videos...
  24,493 clips ready

Trainable parameters: 7,400,839


#### Training loop

In [28]:
print(f"\nStage 8B — Temporal Transformer on ViT-Small features")
print(f"Distillation from Stage 8 MLP + Phase Classification")
print("="*60)

best_val_f1 = 0

for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()

    # train
    model8b.train()
    tr_losses, tr_preds, tr_labels = [], [], []

    for batch_idx, (vit_feats, dino_feats, labels) in \
            enumerate(train_loader):
        vit_feats  = vit_feats.to(device)
        dino_feats = dino_feats.to(device)
        labels     = labels.to(device)

        logits, h, proj = model8b(vit_feats)

        # classification loss
        loss_cls = criterion(logits, labels)

        # distillation — match Transformer output to MLP teacher
        with torch.no_grad():
            B, T, _ = dino_feats.shape
            dino_flat    = dino_feats.view(B*T, -1)
            h_teach, _   = mlp_teacher(dino_flat)
            h_teach      = h_teach.view(B, T, -1)   # (B, T, 384)
            # project teacher to 768 for comparison
            proj_teach   = mlp_teacher.proj(
                h_teach.view(B*T, -1)
            ).view(B, T, -1)                         # (B, T, 768)

        proj_norm  = F.normalize(proj,       dim=-1)
        teach_norm = F.normalize(proj_teach, dim=-1)
        loss_distill = (
            2 - 2 * (proj_norm * teach_norm).sum(dim=-1)
        ).mean()

        loss = CLASS_WEIGHT * loss_cls + \
               DISTILL_WEIGHT * loss_distill

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model8b.parameters(), max_norm=1.0
        )
        optimizer.step()

        tr_losses.append(loss.item())
        tr_preds.extend(logits.argmax(1).cpu().numpy())
        tr_labels.extend(labels.cpu().numpy())

        # timing after first batch
        if batch_idx == 0 and epoch == 1:
            bt = time.time() - t0
            nb = len(train_loader)
            print(f"\n  First batch: {bt:.2f}s | "
                  f"Est. epoch: {bt*nb/60:.1f} min | "
                  f"Est. total: {bt*nb*N_EPOCHS/3600:.1f} hrs\n")

    scheduler.step()
    tr_f1 = f1_score(tr_labels, tr_preds,
                      average='macro', zero_division=0)

    # validate
    model8b.eval()
    val_losses, val_preds, val_labels = [], [], []

    with torch.no_grad():
        for vit_feats, dino_feats, labels in val_loader:
            vit_feats = vit_feats.to(device)
            labels    = labels.to(device)

            logits, _, _ = model8b(vit_feats)
            loss         = criterion(logits, labels)

            val_losses.append(loss.item())
            val_preds.extend(logits.argmax(1).cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_f1   = f1_score(val_labels, val_preds,
                         average='macro', zero_division=0)
    val_loss = np.mean(val_losses)
    elapsed  = time.time() - t0

    print(f"  Epoch {epoch:2d} | "
          f"Train F1: {tr_f1:.3f} | "
          f"Val F1: {val_f1:.3f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"{elapsed:.1f}s")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({
            'epoch':            epoch,
            'model_state_dict': model8b.state_dict(),
            'best_val_f1':      best_val_f1,
            'val_loss':         val_loss,
        }, os.path.join(CHECKPOINT_DIR, 'stage8b_best.pt'))

print(f"\n{'='*60}")
print(f"Stage 8B complete")
print(f"Best Val F1:     {best_val_f1:.3f}")
print(f"Stage 7 scratch: 0.609")
print(f"Stage 8 linear:  0.623")
print(f"Gain over Stage 7: +{best_val_f1 - 0.609:.3f}")


Stage 8B — Temporal Transformer on ViT-Small features
Distillation from Stage 8 MLP + Phase Classification

  First batch: 0.60s | Est. epoch: 3.4 min | Est. total: 1.7 hrs

  Epoch  1 | Train F1: 0.711 | Val F1: 0.680 | Val Loss: 1.0664 | 11.1s
  Epoch  2 | Train F1: 0.859 | Val F1: 0.706 | Val Loss: 1.0174 | 10.1s
  Epoch  3 | Train F1: 0.915 | Val F1: 0.710 | Val Loss: 1.0056 | 10.2s
  Epoch  4 | Train F1: 0.939 | Val F1: 0.687 | Val Loss: 1.1175 | 10.5s
  Epoch  5 | Train F1: 0.958 | Val F1: 0.713 | Val Loss: 1.0374 | 10.3s
  Epoch  6 | Train F1: 0.967 | Val F1: 0.684 | Val Loss: 1.1448 | 10.2s
  Epoch  7 | Train F1: 0.979 | Val F1: 0.687 | Val Loss: 1.1385 | 10.0s
  Epoch  8 | Train F1: 0.985 | Val F1: 0.704 | Val Loss: 1.1017 | 11.5s
  Epoch  9 | Train F1: 0.987 | Val F1: 0.693 | Val Loss: 1.1551 | 10.1s
  Epoch 10 | Train F1: 0.990 | Val F1: 0.702 | Val Loss: 1.1735 | 10.2s
  Epoch 11 | Train F1: 0.994 | Val F1: 0.701 | Val Loss: 1.1383 | 10.5s
  Epoch 12 | Train F1: 0.992 | Va